In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session


# Global YouTube Top 1000 - EDA
Simple and readable analysis for subscribers, views, channels, and topic categories.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
pd.set_option('display.max_colwidth', 120)

DATA_PATH = '/kaggle/input/datasets/mubashirsidiki/youtube-top-1000-global-youtube-channels-by-subscribers/youtube_top_1000_by_subscribers.csv'
df = pd.read_csv(DATA_PATH)

df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(r'[^a-z0-9]+', '_', regex=True)
    .str.strip('_')
)

print('Shape:', df.shape)
print('Columns:', list(df.columns))
display(df.head(3))

In [ ]:
# Data quality snapshot
display(df.info())

quality = pd.DataFrame({
    'missing_values': df.isna().sum(),
    'missing_pct': (df.isna().mean() * 100).round(2),
    'n_unique': df.nunique()
}).sort_values('missing_values', ascending=False)

display(quality)
print('Duplicate rows:', df.duplicated().sum())

In [ ]:
# Type conversion for core metrics
for col in ['rank', 'views', 'subscribers', 'videos']:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

if 'published_at' in df.columns:
    df['published_at'] = pd.to_datetime(df['published_at'], errors='coerce')
    df['publish_year'] = df['published_at'].dt.year

display(df.describe(include='all').T)

In [ ]:
# Top 10 channels by subscribers
top10 = df.nlargest(10, 'subscribers')[['title', 'subscribers']].copy()

# Kaggle default font can miss some glyphs; create safe labels for plotting
top10['title_plot'] = top10['title'].astype(str).str.encode('ascii', errors='ignore').str.decode('ascii').str.strip()
top10['title_plot'] = top10['title_plot'].replace('', np.nan).fillna('Non-Latin Channel')

display(top10[['title', 'subscribers']])

plt.figure(figsize=(10, 5))
sns.barplot(data=top10, x='subscribers', y='title_plot', hue='title_plot', palette='Blues_r', dodge=False, legend=False)
plt.title('Top 10 Channels by Subscribers')
plt.xlabel('Subscribers')
plt.ylabel('Channel')
plt.tight_layout()
plt.show()


In [ ]:
# Metric distributions
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

sns.histplot(df['subscribers'].dropna(), bins=30, ax=axes[0], color='teal')
axes[0].set_title('Subscribers')

sns.histplot(df['views'].dropna(), bins=30, ax=axes[1], color='orange')
axes[1].set_title('Views')

sns.histplot(df['videos'].dropna(), bins=30, ax=axes[2], color='purple')
axes[2].set_title('Videos')

plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap
corr = df[['rank', 'subscribers', 'views', 'videos']].corr(numeric_only=True)
display(corr)

plt.figure(figsize=(6, 4))
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Correlation Heatmap')
plt.tight_layout()
plt.show()

In [ ]:
# Top countries by number of channels
country_counts = df['country'].fillna('Unknown').replace('', 'Unknown').value_counts().head(15)
display(country_counts)

plt.figure(figsize=(10, 6))
sns.barplot(x=country_counts.values, y=country_counts.index, hue=country_counts.index, palette='mako', dodge=False, legend=False)
plt.title('Top 15 Countries by Channel Count')
plt.xlabel('Number of Channels')
plt.ylabel('Country')
plt.tight_layout()
plt.show()

In [ ]:
# Top topic categories
topic_series = (
    df['topic_categories']
    .fillna('')
    .astype(str)
    .str.split(' | ', regex=False)
    .explode()
    .str.strip()
)
topic_series = topic_series[topic_series != '']
top_topics = topic_series.value_counts().head(12)
display(top_topics)

plt.figure(figsize=(11, 6))
sns.barplot(x=top_topics.values, y=top_topics.index, hue=top_topics.index, palette='viridis', dodge=False, legend=False)
plt.title('Top Topic Categories')
plt.xlabel('Count')
plt.ylabel('Topic Category')
plt.tight_layout()
plt.show()

## Summary
- Reviewed data quality and completeness.
- Visualized top channels and distribution patterns.
- Compared countries and topic categories at a glance.